In [0]:
dbutils.widgets.text("catalog_name", "dbw_lakehouse_dev")
dbutils.widgets.text("schema_name", "cloud_practice_bronze")
dbutils.widgets.text("storage_account", "staccount20260516su11dev")
dbutils.widgets.text("container", "lake")
dbutils.widgets.text("landing_prefix", "landing")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
landing_prefix = dbutils.widgets.get("landing_prefix").strip("/")

landing_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{landing_prefix}/"

volume_name = "autoloader_meta"
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

checkpoint_json = f"{volume_path}/checkpoint/orders_json"
schema_json     = f"{volume_path}/schema/orders_json"
checkpoint_csv  = f"{volume_path}/checkpoint/orders_csv"
schema_csv      = f"{volume_path}/schema/orders_csv"

bronze_json_table = f"{catalog_name}.{schema_name}.orders_json"
bronze_csv_table  = f"{catalog_name}.{schema_name}.orders_csv"

print("Landing:", landing_path)
print("Checkpoint JSON:", checkpoint_json)
print("Target table:", bronze_json_table)

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
COMMENT 'Bronze layer – raw ingest from landing'
""")

In [0]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}
COMMENT 'Auto Loader checkpoints and schema inference'
""")

In [0]:
files = dbutils.fs.ls(landing_path)
json_files = [f for f in files if f.name.endswith(".json")]
csv_files = [f for f in files if f.name.endswith(".csv")]

if not json_files and not csv_files:
    raise FileNotFoundError(f"No .json or .csv under {landing_path}")

print(f"Landing: {len(json_files)} json, {len(csv_files)} csv file(s)")

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("pathGlobFilter", "*.json")
    .option("cloudFiles.schemaLocation", schema_json)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_json)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_json_table)
    .awaitTermination()
)

print("Bronze JSON ingest completed.")

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .option("cloudFiles.schemaLocation", schema_csv)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_csv)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_csv_table)
    .awaitTermination()
)

print("Bronze CSV ingest completed.")

In [0]:
json_df = spark.table(bronze_json_table)
csv_df = spark.table(bronze_csv_table)

metrics = {
    "bronze_orders_json_rows": json_df.count(),
    "bronze_orders_csv_rows": csv_df.count(),
    "landing_json_files": len(json_files),
    "landing_csv_files": len(csv_files),
}
print(metrics)

# Optional: fail ADF if nothing ever landed in bronze when files existed
if json_files and metrics["bronze_orders_json_rows"] == 0:
    raise RuntimeError("JSON files in landing but bronze.orders_json is empty")